# VideoToSMPL — Notebook 03: End-to-end pipeline

Single notebook: **video in → G1 PKL + preview MP4 out.**

Equivalent to running Notebook 01 + Notebook 02 back-to-back, but without re-uploading the intermediate `.pt`.

**Runtime:** ~3 min for a 10-second video on T4.

## 1. Setup (runs cells from 01 + 02)

In [ ]:
%cd /content
import subprocess, shutil, sys, torch
assert shutil.which('nvidia-smi') and torch.cuda.is_available(), 'GPU not available'
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader']).decode())

![ -d GVHMR ] || git clone --depth 1 https://github.com/zju3dv/GVHMR.git
![ -d GMR ]   || git clone --depth 1 https://github.com/generalroboticslab/GMR.git
!pip install -q -e GVHMR -r GVHMR/requirements.txt
!pip install -q -e GMR
!pip install -q 'mujoco>=3.1' 'imageio[ffmpeg]' scipy smplx

## 2. Download GVHMR weights

In [ ]:
import urllib.request
from pathlib import Path
CKPT = Path('/content/GVHMR/inputs/checkpoints')
for sub in ['gvhmr','hmr2','vitpose','yolo','body_models/smpl','body_models/smplx']:
    (CKPT/sub).mkdir(parents=True, exist_ok=True)
WEIGHTS = {
  'gvhmr/gvhmr_siga24_release.ckpt':'https://huggingface.co/camenduru/GVHMR/resolve/main/gvhmr/gvhmr_siga24_release.ckpt',
  'hmr2/epoch=10-step=25000.ckpt':'https://huggingface.co/camenduru/GVHMR/resolve/main/hmr2/epoch%3D10-step%3D25000.ckpt',
  'vitpose/vitpose-h-multi-coco.pth':'https://huggingface.co/camenduru/GVHMR/resolve/main/vitpose/vitpose-h-multi-coco.pth',
  'yolo/yolov8x.pt':'https://huggingface.co/camenduru/GVHMR/resolve/main/yolo/yolov8x.pt',
  'body_models/smpl/SMPL_NEUTRAL.pkl':'https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_NEUTRAL.pkl',
}
for rel,url in WEIGHTS.items():
    dest = CKPT/rel
    if dest.exists() and dest.stat().st_size>1024: print(f'✓ cached {dest.name}'); continue
    print(f'↓ {dest.name}'); urllib.request.urlretrieve(url, dest)
print('Weights ready.')

## 3. Upload video

In [ ]:
from google.colab import files
uploaded = files.upload()
video = Path('/content')/list(uploaded.keys())[0]
print(f'{video.name} ({video.stat().st_size/1e6:.1f} MB)')

## 4. Run extraction + retarget

In [ ]:
import subprocess, time
%cd /content/GVHMR
t0 = time.time()
r = subprocess.run(['python','tools/demo/demo.py','--video',str(video),'-s'], capture_output=True, text=True)
print('\n'.join(r.stdout.splitlines()[-10:]))
assert r.returncode == 0, r.stderr
pt = Path('/content/GVHMR/outputs/demo')/video.stem/'hmr4d_results.pt'
print(f'GVHMR: {time.time()-t0:.0f}s')

%cd /content/GMR
pkl = Path('/content')/(video.stem+'_g1.pkl')
t0 = time.time()
r = subprocess.run(['python','scripts/gvhmr_to_robot.py','--gvhmr_pt',str(pt),'--robot','unitree_g1','--save_path',str(pkl)], capture_output=True, text=True)
assert r.returncode == 0, r.stderr
print(f'Retarget: {time.time()-t0:.0f}s')

## 5. Render preview + download

In [ ]:
import pickle, numpy as np, imageio.v2 as imageio, mujoco
from general_motion_retargeting import ROBOT_XML_DICT
with open(pkl,'rb') as f: d = pickle.load(f)
model = mujoco.MjModel.from_xml_path(str(ROBOT_XML_DICT['unitree_g1']))
data, renderer = mujoco.MjData(model), mujoco.Renderer(model, height=720, width=1280)
cam = mujoco.MjvCamera(); cam.distance=3.0; cam.azimuth=90; cam.elevation=-15; cam.lookat[:]=[0,0,0.9]
preview = Path('/content')/(video.stem+'_g1.mp4')
w = imageio.get_writer(str(preview), fps=int(d['fps']), codec='libx264', quality=8)
for i in range(len(d['root_pos'])):
    qw,qx,qy,qz = d['root_rot'][i][[3,0,1,2]]
    data.qpos[:3]=d['root_pos'][i]; data.qpos[3:7]=[qw,qx,qy,qz]; data.qpos[7:]=d['dof_pos'][i]
    mujoco.mj_forward(model,data); renderer.update_scene(data, camera=cam)
    w.append_data(np.asarray(renderer.render()))
w.close(); renderer.close()

from google.colab import files
files.download(str(pt)); files.download(str(pkl)); files.download(str(preview))